# WSL pulse audio attempt

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sounddevice as sd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

class AudioCapture:
    def __init__(self):
        self.fs = 44100  # Sample rate
        self.channels = 2  # Stereo
        self.duration = 5  # Recording duration in seconds
        self.recording = None
        self.is_recording = False
        
        # Create widgets
        self.start_button = widgets.Button(description="Start Recording")
        self.stop_button = widgets.Button(description="Stop Recording")
        self.save_button = widgets.Button(description="Save Recording")
        self.play_button = widgets.Button(description="Play Recording")
        self.device_dropdown = widgets.Dropdown(
            options=[(str(d), d) for d in range(len(sd.query_devices()))],
            description='Audio Device:',
        )
        
        # Set up widget callbacks
        self.start_button.on_click(self.start_recording)
        self.stop_button.on_click(self.stop_recording)
        self.save_button.on_click(self.save_recording)
        self.play_button.on_click(self.play_recording)
        
        # Display widgets
        display(widgets.VBox([
            self.device_dropdown,
            widgets.HBox([self.start_button, self.stop_button, self.save_button, self.play_button])
        ]))
    
    def start_recording(self, b):
        self.is_recording = True
        self.recording = sd.rec(int(self.duration * self.fs), samplerate=self.fs, channels=self.channels, device=self.device_dropdown.value)
        print("Recording started...")
    
    def stop_recording(self, b):
        self.is_recording = False
        sd.stop()
        print("Recording stopped.")
    
    def save_recording(self, b):
        if self.recording is not None:
            sd.write('output.wav', self.recording, self.fs)
            print("Recording saved as 'output.wav'")
        else:
            print("No recording to save.")
    
    def play_recording(self, b):
        if self.recording is not None:
            sd.play(self.recording, self.fs)
            sd.wait()
        else:
            print("No recording to play.")

# Create and display the audio capture interface
audio_capture = AudioCapture()

# Old

In [ ]:
import ui_jupyter
import ipywidgets as widgets

In [ ]:
import pathlib
from IPython.display import Audio, display
import ipywidgets as widgets

def play_audio_from_path(file_path):
    """
    Create an audio player widget for the given audio file path.
    
    :param file_path: Path to the audio file
    :return: IPython Audio widget
    """
    path = pathlib.Path(file_path)
    file_name = path.stem  # Extract file name from path
    audio = Audio(filename=file_path)
    return {
        'name' : file_name, 
        'audio' : audio
    }


In [ ]:
import pyaudio
import wave
import numpy as np
import threading

class MicrophoneRecorder:
    def __init__(self, sample_rate=44100, chunk_size=1024, channels=1):
        self.sample_rate = sample_rate
        self.chunk_size = chunk_size
        self.channels = channels
        self.p = pyaudio.PyAudio()
        self.stream = None
        self.frames = []
        self.is_recording = False

    @staticmethod
    def list_devices():
        p = pyaudio.PyAudio()
        info = p.get_host_api_info_by_index(0)
        num_devices = info.get('deviceCount')
        
        print("Available audio input devices:")
        for i in range(num_devices):
            device_info = p.get_device_info_by_host_api_device_index(0, i)
            if device_info.get('maxInputChannels') > 0:
                print(f"Device ID {i}: {device_info.get('name')}")
        p.terminate()

    def start_recording(self, device_index=None):
        self.is_recording = True
        self.frames = []
        
        def callback(in_data, frame_count, time_info, status):
            self.frames.append(in_data)
            return (in_data, pyaudio.paContinue)

        try:
            self.stream = self.p.open(format=pyaudio.paInt16,
                                      channels=self.channels,
                                      rate=self.sample_rate,
                                      input=True,
                                      input_device_index=device_index,
                                      frames_per_buffer=self.chunk_size,
                                      stream_callback=callback)
            
            self.stream.start_stream()
            print("Recording started...")
        except Exception as e:
            print(f"Error starting recording: {e}")
            self.is_recording = False

    def stop_recording(self):
        if self.stream:
            self.stream.stop_stream()
            self.stream.close()
        self.is_recording = False
        print("Recording stopped.")

    def save_recording(self, filename):
        if not self.frames:
            print("No audio data to save.")
            return

        wf = wave.open(filename, 'wb')
        wf.setnchannels(self.channels)
        wf.setsampwidth(self.p.get_sample_size(pyaudio.paInt16))
        wf.setframerate(self.sample_rate)
        wf.writeframes(b''.join(self.frames))
        wf.close()
        print(f"Recording saved as {filename}")

    def __del__(self):
        self.p.terminate()

In [ ]:
MicrophoneRecorder.list_devices()

In [ ]:
player_info = play_audio_from_path(file_path='test_recording.wav')
display(player_info['audio'])

In [ ]:
# Create and display widgets
mic_widget = ui_jupyter.AudioCaptureWidget(device_type='microphone')
desktop_widget = ui_jupyter.AudioCaptureWidget(device_type='desktop')

display(widgets.HBox([mic_widget.widget, desktop_widget.widget]))